In [1]:
import gzip
from collections import defaultdict
import math
import scipy.optimize
from sklearn import svm
import numpy
import string
import random
import string
from sklearn import linear_model

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [3]:
def assertFloat(x):
    assert type(float(x)) == float

def assertFloatList(items, N):
    assert len(items) == N
    assert [type(float(x)) for x in items] == [float]*N

In [4]:
def readGz(path):
    for l in gzip.open(path, 'rt'):
        yield eval(l)

In [5]:
def readCSV(path):
    f = gzip.open(path, 'rt')
    f.readline()
    for l in f:
        u,b,r = l.strip().split(',')
        r = int(r)
        yield u,b,r

In [6]:
allRatings = []
for l in readCSV("train_Interactions.csv.gz"):
    allRatings.append(l)

In [7]:
ratingsTrain = allRatings[:190000]
ratingsValid = allRatings[190000:]
ratingsPerUser = defaultdict(list)
ratingsPerItem = defaultdict(list)
for u,b,r in ratingsTrain:
    ratingsPerUser[u].append((b,r))
    ratingsPerItem[b].append((u,r))

In [8]:
#Task 1: Predict a given user book pair for pairs read csv, surpise 

In [9]:
bookCount = defaultdict(int)
totalRead = 0

for user,book,_ in readCSV("train_Interactions.csv.gz"):
  bookCount[book] += 1
  totalRead += 1

mostPopular = [(bookCount[x], x) for x in bookCount]
mostPopular.sort()
mostPopular.reverse()

return1 = set()
count = 0
for ic, i in mostPopular:
  count += ic
  return1.add(i)
  if count > totalRead/2: break

In [10]:
#Task 1: generate output 0, 1 from books read or not read 

In [11]:
userSet = set()
bookSet = set()
readSet = set()

for u,b,r in allRatings:
    userSet.add(u)
    bookSet.add(b)
    readSet.add((u,b))

lUserSet = list(userSet)
lBookSet = list(bookSet)

notRead = set()
for u,b,r in ratingsValid:
    #u = random.choice(lUserSet)
    b = random.choice(lBookSet)
    while ((u,b) in readSet or (u,b) in notRead):
        b = random.choice(lBookSet)
    notRead.add((u,b))

readValid = set()
for u,b,r in ratingsValid:
    readValid.add((u,b))

In [12]:
def Jaccard(s1, s2):
    numer = len(s1.intersection(s2))
    denom = len(s1.union(s2))
    if denom > 0:
        return numer/denom
    return 0

In [13]:
correct = 0
p0, p1 = 0,0
for (label,sample) in [(1, readValid), (0, notRead)]:
    for (u,b) in sample:
        pred = 0
        if b in return1:
            pred = 1
        if pred == label:
            correct += 1

In [14]:
correct / (len(readValid) + len(notRead))

0.7144

In [15]:
return1 = set()
count = 0
for ic, i in mostPopular:
    count += ic
    return1.add(i)
    if count > 1.5 * totalRead/2: break

In [16]:
correct = 0
p0, p1 = 0,0
for (label,sample) in [(1, readValid), (0, notRead)]:
    for (u,b) in sample:
        pred = 0
        if b in return1:
            pred = 1
        if pred == label:
            correct += 1

In [17]:
correct / (len(readValid) + len(notRead))

0.75505

In [18]:
correct = 0
for (label,sample) in [(1, readValid), (0, notRead)]:
    for (u,b) in sample:
        maxSim = 0
        users = set(ratingsPerItem[b])
        for b2,_ in ratingsPerUser[u]:
            sim = Jaccard(users,set(ratingsPerItem[b2]))
            if sim > maxSim:
                maxSim = sim
        pred = 0
        if maxSim > 0.013 or len(ratingsPerItem[b]) > 40:
            pred = 1
        if pred == label:
            correct += 1

In [137]:
correct / (len(readValid) + len(notRead))

0.7495

In [138]:
predictions = open("predictions_Read.csv", 'w')
for l in open("pairs_Read.csv"):
  if l.startswith("userID"):
    #header
    predictions.write(l)
    continue
  u,b = l.strip().split(',')
  if b in return1:
    predictions.write(u + ',' + b + ",1\n")
  else:
    predictions.write(u + ',' + b + ",0\n")

predictions.close()

In [139]:
#task 2 ratings predict, use suprise library

In [140]:
allRatings = []
userRatings = defaultdict(list)

for user,book,r in readCSV("train_Interactions.csv.gz"):
  r = int(r)
  allRatings.append(r)
  userRatings[user].append(r)

globalAverage = sum(allRatings) / len(allRatings)
userAverage = {}
for u in userRatings:
  userAverage[u] = sum(userRatings[u]) / len(userRatings[u])

In [141]:
allRatings = []
for l in readCSV("train_Interactions.csv.gz"):
    allRatings.append(l)

In [142]:
ratingsTrain = allRatings[:190000]
ratingsValid = allRatings[190000:]
ratingsPerUser = defaultdict(list)
ratingsPerItem = defaultdict(list)
for u,b,r in ratingsTrain:
    ratingsPerUser[u].append((b,r))
    ratingsPerItem[b].append((u,r))

In [143]:
ratingsValid

[('u47877739', 'b50020691', 4),
 ('u44286663', 'b17025078', 5),
 ('u73526619', 'b22925633', 5),
 ('u39123859', 'b62815615', 0),
 ('u12786783', 'b99699446', 2),
 ('u99072969', 'b24832848', 4),
 ('u84518411', 'b73152433', 2),
 ('u50827374', 'b41385722', 3),
 ('u16732033', 'b77335595', 4),
 ('u24849922', 'b00310513', 5),
 ('u57991147', 'b69655340', 1),
 ('u76612091', 'b85599048', 2),
 ('u38719536', 'b59839746', 5),
 ('u67528277', 'b53393493', 5),
 ('u34737050', 'b39530442', 4),
 ('u29262507', 'b56096640', 4),
 ('u94569961', 'b58807152', 4),
 ('u58715931', 'b82180436', 5),
 ('u51998106', 'b08804125', 0),
 ('u41393840', 'b27008357', 3),
 ('u51142543', 'b86467914', 4),
 ('u66115844', 'b13034223', 4),
 ('u26707024', 'b67596539', 2),
 ('u32255041', 'b23317003', 5),
 ('u33819512', 'b44666390', 5),
 ('u29996833', 'b01497094', 2),
 ('u65645818', 'b89780411', 2),
 ('u93623119', 'b09881661', 4),
 ('u76724157', 'b44553593', 3),
 ('u13875453', 'b05595429', 5),
 ('u28867311', 'b52174951', 3),
 ('u8268

In [144]:
#start

In [145]:
trainRatings = [r[2] for r in ratingsTrain]
globalAverage = sum(trainRatings) * 1.0 / len(trainRatings)

In [146]:
validMSE = 0
for u,b,r in ratingsValid:
  se = (r - globalAverage)**2
  validMSE += se

validMSE /= len(ratingsValid)

print("Test MSE (average only) = " + str(validMSE))

Test MSE (average only) = 1.7272009396120143


In [147]:
betaU = {}
betaI = {}
for u in ratingsPerUser:
    betaU[u] = 0

for b in ratingsPerItem:
    betaI[b] = 0

In [148]:
alpha = globalAverage

In [149]:
def iterate(lamb):
    newAlpha = 0
    for u,b,r in ratingsTrain:
        newAlpha += r - (betaU[u] + betaI[b])
    alpha = newAlpha / len(ratingsTrain)
    for u in ratingsPerUser:
        newBetaU = 0
        for b,r in ratingsPerUser[u]:
            newBetaU += r - (alpha + betaI[b])
        betaU[u] = newBetaU / (lamb + len(ratingsPerUser[u]))
    for b in ratingsPerItem:
        newBetaI = 0
        for u,r in ratingsPerItem[b]:
            newBetaI += r - (alpha + betaU[u])
        betaI[b] = newBetaI / (lamb + len(ratingsPerItem[b]))
    mse = 0
    for u,b,r in ratingsTrain:
        prediction = alpha + betaU[u] + betaI[b]
        mse += (r - prediction)**2
    regularizer = 0
    for u in betaU:
        regularizer += betaU[u]**2
    for b in betaI:
        regularizer += betaI[b]**2
    mse /= len(ratingsTrain)
    return mse, mse + lamb*regularizer

In [150]:
mse,objective = iterate(1)
newMSE,newObjective = iterate(1)
iterations = 2

In [151]:
while iterations < 10 or objective - newObjective > 0.0001:
    mse, objective = newMSE, newObjective
    newMSE, newObjective = iterate(1)
    iterations += 1
    print("Objective after "
        + str(iterations) + " iterations = " + str(newObjective))
    print("MSE after "
        + str(iterations) + " iterations = " + str(newMSE))

Objective after 3 iterations = 13116.588597112237
MSE after 3 iterations = 1.0532999574393143
Objective after 4 iterations = 13126.061979708516
MSE after 4 iterations = 1.0531700104166957
Objective after 5 iterations = 13123.880213845163
MSE after 5 iterations = 1.053139460478934
Objective after 6 iterations = 13119.024310980794
MSE after 6 iterations = 1.0531277418089378
Objective after 7 iterations = 13113.705317224998
MSE after 7 iterations = 1.05312075617977
Objective after 8 iterations = 13108.483752331393
MSE after 8 iterations = 1.0531152532250623
Objective after 9 iterations = 13103.503333711971
MSE after 9 iterations = 1.0531103584002324
Objective after 10 iterations = 13098.795339035578
MSE after 10 iterations = 1.0531058094104893
Objective after 11 iterations = 13094.358965836527
MSE after 11 iterations = 1.0531015180428667
Objective after 12 iterations = 13090.184043329817
MSE after 12 iterations = 1.0530974490161191
Objective after 13 iterations = 13086.257833272284
MSE af

In [152]:
validMSE = 0
for u,b,r in ratingsValid:
    bu = 0
    bi = 0
    if u in betaU:
        bu = betaU[u]
    if b in betaI:
        bi = betaI[b]
    prediction = alpha + bu + bi
    validMSE += (r - prediction)**2

validMSE /= len(ratingsValid)
print("Validation MSE = " + str(validMSE))

Validation MSE = 1.4867655009847975


In [153]:
predictions = open("predictions_Rating.csv", 'w')
for l in open("pairs_Rating.csv"):
    if l.startswith("userID"):
        predictions.write(l)
        continue
    u,b = l.strip().split(',')
    bu = 0
    bi = 0
    if u in betaU:
        bu = betaU[u]
    if b in betaI:
        bi = betaI[b]
        temp = (alpha + bu + bi)
    _ = predictions.write(u + ',' + b + ',' + str(temp) + '\n')

predictions.close()

In [154]:
import pandas as pd

In [155]:
rating = pd.read_csv("predictions_Rating.csv")

In [156]:
rating.describe()

,prediction
count,10000.000000
mean,3.860170
std,0.728941
min,-0.295733
25%,3.452727
50%,3.921824
75%,4.353107
max,5.799050
